# Instalação de dependências e setup do SSD++

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import subprocess, sys, os

# Instalar dependências do SSD++
subprocess.run([sys.executable, "-m", "pip", "install", "gmpy2", "numba",
                "--break-system-packages", "-q"], check=True)

SSDPP_PATH = "./SSDpp-numeric"

if not os.path.exists(SSDPP_PATH):
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HMProenca/SSDpp-numeric.git", SSDPP_PATH],
        check=True
    )

# Correção de incompatibilidade com pandas >= 2.0
rd_path = f"{SSDPP_PATH}/src/util/_read_dataset.py"
content = open(rd_path).read()
patches = [
    ("df.loc[:,colname] = df.loc[:,colname] .astype('category')",
     "df[colname] = df[colname].astype(str).astype('category')"),
    ("categories = list(df.loc[:,colname].cat.categories)",
     "categories = list(df[colname].cat.categories)"),
    ("df.loc[:,colname] = df.loc[:,colname].cat.codes",
     "df[colname] = df[colname].cat.codes"),
    ("df.loc[:,colname]  = df[colname].astype('category')",
     "df[colname] = df[colname].astype(str).astype('category')"),
]
for old, new in patches:
    content = content.replace(old, new)
open(rd_path, "w").write(content)

if SSDPP_PATH not in sys.path:
    sys.path.insert(0, SSDPP_PATH)

print("Setup concluído.")

# Função para plotar mapas

In [ ]:
import plotly.express as px

def plot_presentation_map_large_text(df, group_column='group', output_filename='us_classification_map.png'):
    # 1. Create the base geographical scatter plot
    fig = px.scatter_geo(
        df,
        lat='latitude',
        lon='longitude',
        color=group_column,
        scope='usa',
        projection='albers usa', 
        title=f'<b>Spatial Distribution Analysis: {group_column.replace("_", " ").title()}</b>',
        color_discrete_sequence=px.colors.qualitative.Dark24
    )

    # 2. Refine the visual theme with extra-large typography
    fig.update_layout(
        title_font=dict(size=34, family="Helvetica, Arial", color="#1a252f"),
        paper_bgcolor='white',
        margin=dict(l=10, r=10, t=90, b=10),
        
        # Legend styling - Upgraded for independent marker scaling
        legend=dict(
            title_text=f"<b>{group_column.replace('_', ' ').title()}</b>",
            title_font=dict(size=26),  
            font=dict(size=22),        
            orientation="v",
            yanchor="top",
            y=0.85,
            xanchor="left",
            x=1.02,
            
            # Forces legend icons to be a large, constant size regardless of how tiny map points are
            itemsizing='constant' 
        ),
        
        geo=dict(
            lakecolor='rgb(255, 255, 255)',
            landcolor='rgb(245, 246, 248)',
            subunitcolor='rgb(210, 214, 219)', 
            countrycolor='rgb(160, 165, 175)',
            showlakes=True,
            showsubunits=True,
            showcountries=True
        )
    )

    # 3. Make the map points small/crisp without ruining the legend
    fig.update_traces(
        marker=dict(size=6, opacity=0.80, line=dict(width=0.4, color='White'))
    )

    # 4. Export at high resolution
    print(f"Saving high-res map with decoupled legend size to {output_filename}...")
    fig.write_image(output_filename, width=1920, height=1080, scale=3)

# Experimento 1

In [ ]:
from _classes import SSDC  # SSD++

# 1. Carregar a base de dados
df = pd.read_csv('dataset_clima.csv')

# 2. Mover a coluna 'elevation_meters' para o final do DataFrame para ela ser o alvo
alvo = df.pop('elevation_meters')
df['elevation_meters'] = alvo

# Remover colunas indesejadas
df = df.drop(columns=["point", "sigma2.freq_seasonal_365(1)", "longitude", "latitude"], errors='ignore')

# 3. Definir os parâmetros do experimento
disc_type = "dynamic"
task_name = "discovery"
max_len = 8
beamsize = 100
ncutpoints = 10
iterative = 1
target_type = "numeric"
gain_type = "absolute"

# 4. Inicializar o modelo SSD++
model = SSDC(
    target_type=target_type,
    max_depth=max_len, 
    beam_width=beamsize,
    iterative_beam_width=iterative,
    n_cutpoints=ncutpoints,
    task=task_name,
    discretization=disc_type, 
    gain=gain_type
)

# 5. Executar o algoritmo
print("Iniciando a Descoberta de Subgrupos. Isso pode levar alguns minutos...")
model.fit(df)

## Plot do mapa

In [ ]:
df = pd.read_csv('dataset_clima.csv')

# Custom function: 2 decimals normally, scientific notation for tiny numbers
def custom_formatter(x):
    if pd.isna(x):
        return ""
    if abs(x) == 0:
        return "0.00"
    elif abs(x) < 0.01:
        return f"{x:.2e}"  # 2 decimal places in scientific notation
    else:
        return f"{x:.2f}"  # 2 decimal places normally


In [ ]:
def classify_model_five(df):
    # Sequential condition tracking for Model 5
    conditions = [
        # Rule 1
        (df['total_precip_mm'] >= 26.03545188903809) & (df['total_precip_mm'] <= 57.02376937866211) & 
        (df['longitude'] >= -100.5),
        
        # Rule 2
        (df['longitude'] <= -100.75) & 
        (df['seasonal_amplitude'] >= 9.030269035681384) & (df['seasonal_amplitude'] <= 15.423795947380787) & 
        (df['baseline_temp'] <= 17.808198601338056),
        
        # Rule 3
        (df['total_precip_mm'] >= 53.492095947265625) & 
        (df['baseline_temp'] >= 17.808749579986344),
        
        # Rule 4
        (df['total_precip_mm'] >= 16.30215072631836) & 
        (df['seasonal_amplitude'] >= 10.83598600997977) & 
        (df['seasonal_peak_day'] <= 205.0) & 
        (df['sigma2.level'] >= 4.057010269063492),
        
        # Rule 5
        (df['longitude'] <= -116.25) & 
        (df['Population_Density'] >= -9.371502684418026) & 
        (df['latitude'] >= 36.0) & 
        (df['baseline_temp'] >= 10.273376048868116),
        
        # Rule 6
        (df['seasonal_peak_day'] >= 208.0) & 
        (df['seasonal_amplitude'] <= 10.835683801587953),
        
        # Rule 7
        (df['seasonal_peak_day'] >= 202.0) & 
        (df['latitude'] >= 37.75) & (df['latitude'] <= 41.5),
        
        # Rule 8
        (df['longitude'] >= -116.0) & (df['longitude'] <= -81.25) & 
        (df['baseline_temp'] <= 19.92047601504268) & 
        (df['seasonal_peak_day'] <= 205.0),
        
        # Rule 9
        (df['baseline_temp'] >= 17.808749579986344) & 
        (df['Population_Density'] >= -13.099738005017048) & 
        (df['seasonal_peak_day'] >= 200.0) & 
        (df['sigma2.level'] <= 11.734174002936872),
        
        # Rule 10
        (df['seasonal_amplitude'] >= 10.83598600997977) & (df['seasonal_amplitude'] <= 12.36022004137675) & 
        (df['Population_Density'] >= -8.4493492545818),
        
        # Rule 11
        (df['Population_Density'] >= -12.188035810478747) & (df['Population_Density'] <= -8.45036333823623) & 
        (df['latitude'] >= 36.0) & 
        (df['baseline_temp'] >= 10.273376048868116),
        
        # Rule 12
        df['latitude'] <= 41.5,
        
        # Rule 13
        (df['sigma2.level'] <= 4.05506841885463) & 
        (df['Population_Density'] >= -11.131662721542144) & 
        (df['seasonal_peak_day'] <= 185.0) & 
        (df['sigma2.irregular'] <= 5.644569216297157e-09),
        
        # Rule 14
        df['Population_Density'] >= -10.098246193286904,
        
        # Rule 15
        (df['sigma2.level'] >= 4.057010269063492) & (df['sigma2.level'] <= 9.602406192942505),
        
        # Rule 16
        (df['seasonality_index'] >= 1.2271298170089722) & (df['seasonality_index'] <= 1.4778114557266235)
    ]
    
    # Map elements sequentially to Model5_Group_1 through Model5_Group_16
    choices = [f"Group_{i}" for i in range(1, 17)]
    
    # Final default assignment maps the fallback execution block (Group 17)
    df['model_5_group'] = np.select(conditions, choices, default="Group_17")
    
    return df

df_classified = classify_model_five(df_classified)
df_classified.head()

In [ ]:
plot_presentation_map_large_text(df_classified, group_column='model_5_group', output_filename='exp_1.png')

# Experimento 2

In [ ]:
from _classes import SSDC  # SSD++

# 1. Carregar a base de dados
df = pd.read_csv('dataset_clima.csv')

# 2. Mover a coluna 'elevation_meters' para o final do DataFrame para ela ser o alvo
alvo = df.pop('elevation_meters')
df['elevation_meters'] = alvo

# Remover colunas indesejadas
df = df.drop(columns=["point", "sigma2.freq_seasonal_365(1)"], errors='ignore')

# 3. Definir os parâmetros do experimento
disc_type = "dynamic"
task_name = "discovery"
max_len = 8
beamsize = 100
ncutpoints = 10
iterative = 1
target_type = "numeric"
gain_type = "absolute"

# 4. Inicializar o modelo SSD++
model = SSDC(
    target_type=target_type,
    max_depth=max_len, 
    beam_width=beamsize,
    iterative_beam_width=iterative,
    n_cutpoints=ncutpoints,
    task=task_name,
    discretization=disc_type, 
    gain=gain_type
)

# 5. Executar o algoritmo
print("Iniciando a Descoberta de Subgrupos. Isso pode levar alguns minutos...")
model.fit(df)

In [ ]:
print(model)

## Plot do mapa

In [ ]:
df = pd.read_csv('dataset_clima.csv')

# Custom function: 2 decimals normally, scientific notation for tiny numbers
def custom_formatter(x):
    if pd.isna(x):
        return ""
    if abs(x) == 0:
        return "0.00"
    elif abs(x) < 0.01:
        return f"{x:.2e}"  # 2 decimal places in scientific notation
    else:
        return f"{x:.2f}"  # 2 decimal places normally


In [ ]:
def classify_model_eight(df):
    # Sequential condition tracking for Model 8
    conditions = [
        # Rule 1
        (df['total_precip_mm'] >= 26.03545188903809) & 
        (df['baseline_temp'] >= 9.08767700961204) & 
        (df['sigma2.level'] >= 7.217154289127204) & 
        (df['seasonal_amplitude'] >= 11.659237606411397),
        
        # Rule 2
        (df['seasonal_peak_day'] <= 197.0) & 
        (df['seasonal_amplitude'] >= 9.030269035681384) & (df['seasonal_amplitude'] <= 15.423795947380787) & 
        (df['baseline_temp'] <= 17.808198601338056) & 
        (df['sigma2.irregular'] <= 0.4306638500362468) & 
        (df['total_precip_mm'] <= 45.1981086730957),
        
        # Rule 3
        (df['total_precip_mm'] >= 26.03545188903809) & 
        (df['baseline_temp'] >= 17.808749579986344) & 
        (df['seasonal_peak_day'] >= 186.0),
        
        # Rule 4
        (df['total_precip_mm'] >= 16.30215072631836) & \
        (df['sigma2.level'] >= 12.929430834722524) & \
        (df['seasonal_amplitude'] >= 12.363509175046298) & \
        (df['sigma2.irregular'] >= 6.863613528311764e-10),
        
        # Rule 5
        (df['seasonal_peak_day'] >= 202.0) & \
        (df['seasonality_index'] >= 1.3027167320251465) & \
        (df['Population_Density'] >= -13.099738005017048) & \
        (df['total_precip_mm'] <= 57.02376937866211),
        
        # Rule 6
        (df['seasonal_peak_day'] <= 180.0) & \
        (df['sigma2.level'] <= 4.05506841885463) & \
        (df['Population_Density'] >= -11.622088948620483),
        
        # Rule 7
        (df['baseline_temp'] >= 9.08767700961204) & \
        (df['seasonal_amplitude'] >= 10.83598600997977) & (df['seasonal_amplitude'] <= 15.423795947380787) & \
        (df['seasonal_peak_day'] >= 186.0) & (df['seasonal_peak_day'] <= 207.0) & \
        (df['sigma2.level'] >= 4.057010269063492),
        
        # Rule 8
        (df['total_precip_mm'] >= 26.03545188903809) & (df['total_precip_mm'] <= 45.1981086730957) & \
        (df['seasonal_amplitude'] >= 9.030269035681384) & \
        (df['sigma2.level'] <= 14.950821930702691) & \
        (df['baseline_temp'] <= 11.254375468130446) & \
        (df['seasonal_peak_day'] >= 193.0),
        
        # Rule 9
        (df['baseline_temp'] >= 10.273376048868116) & (df['baseline_temp'] <= 16.023476097239758) & \
        (df['seasonal_amplitude'] <= 9.028118926792716) & \
        (df['Population_Density'] >= -10.65240983856905) & \
        (df['seasonality_index'] <= 1.3594768047332764),
        
        # Rule 10
        (df['seasonality_index'] <= 1.5540281534194946) & \
        (df['total_precip_mm'] <= 26.032581329345703) & \
        (df['baseline_temp'] <= 19.92047601504268) & \
        (df['sigma2.level'] <= 18.269400593238085),
        
        # Rule 11
        (df['baseline_temp'] >= 17.808749579986344) & \
        (df['seasonal_peak_day'] >= 200.0) & \
        (df['sigma2.irregular'] >= 3.511575248596965e-11) & (df['sigma2.irregular'] <= 5.644569216297157e-09) & \
        (df['seasonal_amplitude'] <= 10.835683801587953) & \
        (df['seasonality_index'] >= 1.3027167320251465),
        
        # Rule 12
        (df['seasonal_peak_day'] >= 186.0) & (df['seasonal_peak_day'] <= 202.0) & \
        (df['seasonal_amplitude'] >= 11.659237606411397) & (df['seasonal_amplitude'] <= 15.423795947380787) & \
        (df['Population_Density'] >= -8.4493492545818) & \
        (df['sigma2.irregular'] >= 6.863613528311764e-10) & (df['sigma2.irregular'] <= 1.3530475654278881e-08),
        
        # Rule 13
        (df['Population_Density'] >= -13.099738005017048) & \
        (df['seasonal_peak_day'] >= 208.0) & \
        (df['seasonality_index'] >= 1.2728649377822876) & \
        (df['sigma2.irregular'] >= 1.977104397366684e-09) & (df['sigma2.irregular'] <= 9.077695306719389e-08),
        
        # Rule 14
        (df['seasonal_peak_day'] >= 186.0) & \
        (df['seasonal_amplitude'] >= 11.659237606411397) & \
        (df['sigma2.level'] >= 4.057010269063492) & (df['sigma2.level'] <= 12.925830857035274) & \
        (df['total_precip_mm'] >= 42.60811614990234),
        
        # Rule 15
        (df['baseline_temp'] <= 14.152313560376948) & \
        (df['seasonal_amplitude'] >= 9.030269035681384) & (df['seasonal_amplitude'] <= 12.36022004137675) & \
        (df['sigma2.irregular'] <= 9.077695306719389e-08),
        
        # Rule 16
        (df['seasonal_peak_day'] >= 186.0) & (df['seasonal_peak_day'] <= 202.0) & \
        (df['seasonal_amplitude'] >= 11.659237606411397),
        
        # Rule 17
        (df['baseline_temp'] >= 16.025189136012216) & (df['baseline_temp'] <= 17.808198601338056) & \
        (df['seasonal_peak_day'] >= 193.0) & (df['seasonal_peak_day'] <= 201.0),
        
        # Rule 18
        (df['seasonal_amplitude'] >= 9.030269035681384) & (df['seasonal_amplitude'] <= 10.835683801587953) & \
        (df['total_precip_mm'] >= 16.30215072631836) & (df['total_precip_mm'] <= 50.600582122802734),
        
        # Rule 19
        (df['Population_Density'] >= -13.099738005017048) & \
        (df['seasonal_amplitude'] <= 11.659222023418003),
        
        # Rule 20
        (df['sigma2.level'] >= 4.057010269063492) & \
        (df['baseline_temp'] <= 19.92047601504268),
        
        # Rule 21
        df['seasonal_peak_day'] >= 193.0
    ]
    
    # Track classifications from Model8_Group_1 through Model8_Group_21
    choices = [f"Group_{i}" for i in range(1, 22)]
    
    # Fallback to the final execution block (Group 22)
    df['model_8_group'] = np.select(conditions, choices, default="Group_22")
    
    return df

df_classified = classify_model_eight(df)
df_classified.head()

In [ ]:
plot_presentation_map_large_text(df_classified, group_column='model_8_group', output_filename='exp_2.png')

# Experimento 3

In [ ]:
from _classes import SSDC  # SSD++

# 1. Carregar a base de dados
df = pd.read_csv('dataset_clima.csv')

# 2. Mover a coluna 'Population_Density' para o final do DataFrame para ela ser o alvo
alvo = df.pop('Population_Density')
df['Population_Density'] = alvo

# Remover colunas indesejadas
df = df.drop(columns=["point", "sigma2.irregular", "sigma2.level",
                       "sigma2.freq_seasonal_365(1)"], errors='ignore')

# 3. Definir os parâmetros do experimento
disc_type = "static"
task_name = "discovery"
max_len = 8
beamsize = 100
ncutpoints = 10
iterative = 1
target_type = "numeric"
gain_type = "absolute"

# 4. Inicializar o modelo SSD++
model = SSDC(
    target_type=target_type,
    max_depth=max_len, 
    beam_width=beamsize,
    iterative_beam_width=iterative,
    n_cutpoints=ncutpoints,
    task=task_name,
    discretization=disc_type, 
    gain=gain_type
)

# 5. Executar o algoritmo
print("Iniciando a Descoberta de Subgrupos. Isso pode levar alguns minutos...")
model.fit(df)

Setup concluído.
Iniciando a Descoberta de Subgrupos. Isso pode levar alguns minutos...
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
Iteration: 9
Iteration: 10
Iteration: 11
Iteration: 12
Iteration: 13
Iteration: 14
Iteration: 15


In [6]:
print(model)

IF x in seasonal_peak_day <= 205.0 AND elevation_meters >= 128.0 AND seasonal_amplitude >= 9.030269035681384 AND baseline_temp <= 19.92047601504268 THEN mean = 0.00019722679331664596; std = 0.0005756724474812132 ,  
ELSE IF x in 38.41028594970703 <= total_precip_mm <= 50.600582122802734 AND 1.340889573097229 <= seasonality_index <= 1.3835132122039795 AND elevation_meters <= 19.0 AND baseline_temp >= 11.255998586219304 AND seasonal_peak_day >= 206.0 THEN mean = 0.007427907304999653; std = 0.020236712002572085 ,  
ELSE IF x in -116.0 <= longitude <= -74.75 AND latitude <= 38.75 AND seasonality_index >= 1.2271298170089722 THEN mean = 0.0003054240091647165; std = 0.0006863032978738867 ,  
ELSE IF x in total_precip_mm >= 42.60811614990234 AND baseline_temp <= 12.036866145553638 AND seasonal_peak_day >= 181.0 THEN mean = 0.00039956067510408557; std = 0.0008170407599830226 ,  
ELSE IF x in elevation_meters <= 70.0 AND total_precip_mm >= 45.2022819519043 AND 181.0 <= seasonal_peak_day <= 207.0

## Plot do mapa

In [ ]:
df = pd.read_csv('dataset_clima.csv')

# Custom function: 2 decimals normally, scientific notation for tiny numbers
def custom_formatter(x):
    if pd.isna(x):
        return ""
    if abs(x) == 0:
        return "0.00"
    elif abs(x) < 0.01:
        return f"{x:.2e}"  # 2 decimal places in scientific notation
    else:
        return f"{x:.2f}"  # 2 decimal places normally


In [ ]:
def classify_model_1(df):
    # Define the list of conditional masks
    conditions = [
        # Rule 1
        (df['seasonal_peak_day'] <= 205.0) & \
        (df['elevation_meters'] >= 128.0) & \
        (df['seasonal_amplitude'] >= 9.030269035681384) & \
        (df['baseline_temp'] <= 19.92047601504268),
        
        # Rule 2
        (df['total_precip_mm'] >= 38.41028594970703) & (df['total_precip_mm'] <= 50.600582122802734) & \
        (df['seasonality_index'] >= 1.340889573097229) & (df['seasonality_index'] <= 1.3835132122039795) & \
        (df['elevation_meters'] <= 19.0) & \
        (df['baseline_temp'] >= 11.255998586219304) & \
        (df['seasonal_peak_day'] >= 206.0),
        
        # Rule 3
        (df['longitude'] >= -116.0) & (df['longitude'] <= -74.75) & \
        (df['latitude'] <= 38.75) & \
        (df['seasonality_index'] >= 1.2271298170089722),
        
        # Rule 4
        (df['total_precip_mm'] >= 42.60811614990234) & \
        (df['baseline_temp'] <= 12.036866145553638) & \
        (df['seasonal_peak_day'] >= 181.0),
        
        # Rule 5
        (df['elevation_meters'] <= 70.0) & \
        (df['total_precip_mm'] >= 45.2022819519043) & \
        (df['seasonal_peak_day'] >= 181.0) & (df['seasonal_peak_day'] <= 207.0),
        
        # Rule 6
        (df['seasonal_peak_day'] >= 181.0) & (df['seasonal_peak_day'] <= 199.0) & \
        (df['latitude'] >= 37.75) & (df['latitude'] <= 42.75) & \
        (df['elevation_meters'] >= 20.0),
        
        # Rule 7
        (df['elevation_meters'] <= 314.0) & \
        (df['baseline_temp'] >= 10.273376048868116) & (df['baseline_temp'] <= 19.92047601504268),
        
        # Rule 8
        (df['elevation_meters'] <= 500.0) & \
        (df['seasonal_peak_day'] <= 207.0) & \
        (df['longitude'] <= -78.0) & \
        (df['seasonal_amplitude'] <= 13.874719749694236),
        
        # Rule 9
        (df['longitude'] >= -116.0) & (df['longitude'] <= -81.25) & \
        (df['seasonal_amplitude'] <= 13.174965977800026),
        
        # Rule 10
        (df['latitude'] >= 32.0) & (df['latitude'] <= 33.75) & \
        (df['seasonal_peak_day'] <= 201.0),
        
        # Rule 11
        df['seasonal_peak_day'] <= 197.0,
        
        # Rule 12
        (df['seasonal_peak_day'] >= 202.0) & \
        (df['seasonality_index'] >= 1.322644591331482) & \
        (df['baseline_temp'] >= 9.08767700961204),
        
        # Rule 13
        (df['elevation_meters'] >= 128.0) & (df['elevation_meters'] <= 882.0) & \
        (df['baseline_temp'] >= 9.08767700961204) & (df['baseline_temp'] <= 17.808198601338056),
        
        # Rule 14
        df['latitude'] <= 44.5
    ]
    
    # Map each corresponding rule to a group identifier (Groups 1 to 14)
    choices = [f"Group_{i}" for i in range(1, 15)]
    
    # Apply conditions; default fallback represents the final ELSE branch (Group 15)
    df['model_1_group'] = np.select(conditions, choices, default="Group_15")
    return df

df_classified = classify_model_1(df)
df_classified.head()

,point,latitude,longitude,sigma2.irregular,sigma2.level,sigma2.freq_seasonal_365(1),baseline_temp,seasonal_amplitude,seasonal_peak_day,total_precip_mm,seasonality_index,Population_Density,elevation_meters,model_1_group
0,0,45.75,-116.25,4.248111e-08,6.974031,1.372478e-10,7.975451,11.341726,180,35.971947,1.198418,4.628783e-07,913.0,Group_1
1,1,43.75,-112.00,3.006277e-10,9.712254,3.223228e-10,7.522495,14.561481,181,16.017576,1.301663,7.042701e-05,1464.0,Group_1
2,2,44.75,-115.25,1.299604e-09,7.673272,5.447926e-08,2.762211,11.633025,184,32.565693,1.200082,3.534187e-07,1904.0,Group_1
3,3,44.00,-116.50,2.185260e-11,6.934270,1.072004e-08,10.621284,13.965891,180,17.090015,1.442161,2.853229e-05,902.0,Group_1
4,4,42.50,-114.25,1.974195e-10,8.560141,1.870591e-08,10.235773,13.719436,184,15.187204,1.391501,3.723481e-05,1310.0,Group_1


In [ ]:
plot_presentation_map_large_text(df_classified, group_column='model_1_group', output_filename='exp_3.png')